## baseline physics model


In [1]:
from utils import calculate_nse, create_lag, create_lag_adv

In [2]:
import pandas as pd
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
pd.to_datetime(df['date'])
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.drop(columns= 'Unnamed: 0.1', inplace= True)

df.head()

,dewpoint,et_loss,ndvi,precipitation,radiation,sca,dd,temp,runoff
date,,,,,,,,,
2000-06-30,270.274111,0.035384,0.167895,1.081265,8552.215167,15695.091988,0.000000,255.924696,0.959289
2000-07-31,275.634907,0.005471,0.214835,2.074423,8945.476526,6188.434963,0.000000,253.865719,0.515886
2000-08-31,273.643363,0.016486,0.171795,1.208544,8787.250911,10114.130241,0.000000,258.913569,0.355015
2000-09-30,266.780844,0.040531,0.148384,0.857759,7687.016192,12888.810599,0.000000,265.094486,0.186357
2000-10-31,259.412862,0.029951,0.113129,0.231264,5685.865842,19583.623736,0.551964,273.551964,0.138155


In [3]:
df1 = df.copy()

In [4]:
#for single column dataframes always use double brackets, single brackets will make it a series, or float type
# for adding two columns to a df either add them one by one or use two brackets

### reduce to a single function to test multiple combinations of lag

In [5]:
#train and val script
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
lag_scaler = StandardScaler()


x_lagged = create_lag(df1, lag_precip=1, lag_dd=3, lag_sca=3, lag_et=1)

x_train = x_lagged[:'2017-12-31']
x_val = x_lagged['2018-01-01':'2021-12-31']
x_test = x_lagged['2022-01-01':]


X_train_lagged = x_train[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_train_lagged = x_train[['runoff']]

X_val_lagged = x_val[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_val_lagged = x_val[['runoff']]

X_test_lagged = x_test[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_test_lagged = x_test[['runoff']]

X_train_val = pd.concat([X_train_lagged, X_val_lagged])
Y_train_val = pd.concat([Y_train_lagged, Y_val_lagged])

lag_scaler.fit(X_train_lagged)
 
x_train_scaled = lag_scaler.transform(X_train_lagged)
x_val_scaled = lag_scaler.transform(X_val_lagged)
x_test_scaled = lag_scaler.transform(X_test_lagged)

amodel = LinearRegression()
amodel.fit(X_train_lagged, Y_train_lagged)
# amodel.fit(X_val_lagged, Y_val_lagged)

y_val_pred = amodel.predict(X_val_lagged)
baseline_nse = calculate_nse(Y_val_lagged, y_val_pred)
print(baseline_nse)
# alpha, beta, gamma, delta=amodel.coef_
print(amodel.coef_)

runoff    0.415943
dtype: float64
[[ 3.01059618e-01  7.78374296e-02 -2.00126628e-06  1.43705861e+01]]


/Users/abhimanyu/Desktop/projects/the_indus_project/gisenv/lib/python3.13/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
lag_scaler = StandardScaler()


x_lagged = create_lag(df1, lag_precip=1, lag_dd=3, lag_sca=3, lag_et=1)

x_train = x_lagged[:'2017-12-31']
x_val = x_lagged['2018-01-01':'2021-12-31']
x_test = x_lagged['2022-01-01':]

y_train = x_lagged[:'2017-12-31']
y_val = x_lagged['2018-01-01':'2021-12-31']
y_test = x_lagged['2022-01-01':]

X_train_lagged = x_train[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_train_lagged = y_train[['runoff']]
Y_val_lagged = y_val[['runoff']]
X_val_lagged = x_val[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
X_test_lagged = x_test[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_test_lagged = x_test[['runoff']]

X_train_val = pd.concat([X_train_lagged, X_val_lagged])
Y_train_val = pd.concat([Y_train_lagged, Y_val_lagged])

lag_scaler.fit(X_train_lagged)
 
x_train_scaled = lag_scaler.transform(X_train_lagged)
x_val_scaled = lag_scaler.transform(X_val_lagged)
x_test_scaled = lag_scaler.transform(X_test_lagged)

model = LinearRegression()
model.fit(X_train_val, Y_train_val)
# amodel.fit(X_val_lagged, Y_val_lagged)

y_test_pred = model.predict(X_test_lagged)
baseline_nse = calculate_nse(Y_test_lagged, y_test_pred)
print(baseline_nse)
# alpha, beta, gamma, delta=amodel.coef_
print(model.coef_)

runoff    0.449818
dtype: float64
[[ 2.80571394e-01  1.01534946e-01 -1.88626567e-06  1.32180898e+01]]


/Users/abhimanyu/Desktop/projects/the_indus_project/gisenv/lib/python3.13/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [11]:
# Y_full = pd.concat([Y_train_val, Y_test_lagged])
# df_full = pd.concat([X_train_val, X_test_lagged])
df_full = X_train_val.copy()
df_full['baseline_q'] = model.predict(df_full)
df_full['residuals'] = Y_train_val['runoff'] - df_full['baseline_q']

df_full.to_csv('baseline_data_with_predictions.csv')

df_full_clean = df_full.dropna()

df_adv = create_lag_adv(df1)
df_adv['baseline_q'] = df_full['baseline_q']
df_adv['residuals'] = df_full['residuals']

/Users/abhimanyu/Desktop/projects/the_indus_project/gisenv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:295: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/abhimanyu/Desktop/projects/the_indus_project/gisenv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:295: RuntimeWarning: overflow encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/abhimanyu/Desktop/projects/the_indus_project/gisenv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:295: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_.T + self.intercept_
/var/folders/ql/hdbyk3lx6vq0x6fs9227fc_m0000gn/T/ipykernel_7314/4108333589.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus

In [ ]:

# --- FIX DATA LEAKAGE: Generate baseline predictions without leakage ---

from sklearn.linear_model import LinearRegression

# 1. Train a model ONLY on the training set
model_train_only = LinearRegression()
model_train_only.fit(X_train_lagged, Y_train_lagged)

# 2. Generate predictions for each set using the train-only model
# These are "out-of-sample" predictions for val and test sets.
train_preds = model_train_only.predict(X_train_lagged)
val_preds = model_train_only.predict(X_val_lagged)
test_preds = model_train_only.predict(X_test_lagged)

# 3. Re-combine the predictions and true values in the correct order
baseline_q_no_leak = pd.concat([
    pd.DataFrame(train_preds, index=X_train_lagged.index, columns=['baseline_q']),
    pd.DataFrame(val_preds, index=X_val_lagged.index, columns=['baseline_q']),
    pd.DataFrame(test_preds, index=X_test_lagged.index, columns=['baseline_q'])
])
Y_full = pd.concat([Y_train_val, Y_test_lagged])

# 4. Create the new df_full with non-leaky predictions
df_full_fixed = pd.concat([X_train_lagged, X_val_lagged, X_test_lagged])
df_full_fixed['baseline_q'] = baseline_q_no_leak
df_full_fixed['residuals'] = Y_full['runoff'] - df_full_fixed['baseline_q']

# 5. Now create df_adv using the fixed data
# Assuming create_lag_adv adds new features from df1
df_adv_fixed = create_lag_adv(df1) 
# Align and add the non-leaky columns
df_adv_fixed = df_adv_fixed.join(df_full_fixed[['baseline_q', 'residuals']]).dropna()

In [19]:
from xgboost import XGBRegressor
import pandas as pd

# 1. Re-define the model
xgb = XGBRegressor(
    max_depth=5,
    learning_rate=0.05,
    n_estimators=1500,
    random_state=42, # Add for reproducibility
)

# 2. ALWAYS create your data splits from the correct, non-leaky dataframe in the same cell
train_ml = df_adv_fixed[:'2017-12-31']
val_ml = df_adv_fixed['2018-01-01':'2021-12-31']
test_ml = df_adv_fixed['2022-01-01':]

# 3. Prepare features and labels for the validation run
X_train_ml = train_ml.drop(columns=['baseline_q', 'residuals'])
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(columns=['baseline_q', 'residuals'])
y_val_ml = val_ml['residuals']

# 4. Fit the model ONLY on the training data
xgb.fit(X_train_ml, y_train_ml)

# 5. Predict on the validation set
residual_pred_val = xgb.predict(X_val_ml)
q_hybrid_val = residual_pred_val + val_ml['baseline_q']

# 6. Evaluate against the correct validation ground truth
y_val_true = Y_full['runoff'].loc[val_ml.index]
hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_val)

print(f"Corrected Hybrid NSE (Validation): {hybrid_nse_val:.4f}")

# --- Optional: Final Test Set Evaluation ---
# Now that we've tuned/validated, we can train on train+val and evaluate on test

# Prepare combined training/validation data
X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
y_train_val_ml = pd.concat([y_train_ml, y_val_ml])

# Prepare test data
X_test_ml = test_ml.drop(columns=['baseline_q', 'residuals'])

# Re-fit the model on all available training data
xgb.fit(X_train_val_ml, y_train_val_ml)

# Predict on the test set
residual_pred_test = xgb.predict(X_test_ml)
q_hybrid_test = residual_pred_test + test_ml['baseline_q']

# Evaluate against the test ground truth
y_test_true = Y_full['runoff'].loc[test_ml.index]
hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_test)

print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")


Corrected Hybrid NSE (Validation): 0.9042
Final Hybrid NSE (Test): 0.9147


In [20]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    max_depth  = 5,
    learning_rate = 0.05,
    n_estimators = 1500,
)

# ✅ USE THE CLEANED DATAFRAME
# train_ml = df_adv[:'2017-12-31']
# val_ml = df_adv['2018-01-01':'2021-12-31']
# test_ml = df_adv['2022-01-01':]

# Train XGBoost on residuals
X_train_ml = train_ml.drop(['residuals'], axis = 1)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(['residuals'], axis = 1)
y_val_ml = val_ml['residuals'] # Use the residuals from the same clean dataframe

xgb.fit(X_train_ml, y_train_ml)
residual_pred = xgb.predict(X_val_ml)
q_hybrid = residual_pred + val_ml['baseline_q']

# ✅ IMPORTANT: Use the correct ground truth for comparison
# Y_val_lagged has the wrong index. We need the 'runoff' that corresponds to val_ml.
# We can get this from the original Y_full dataframe by slicing it with the val_ml index.
y_val_true = x_lagged['runoff'].loc[val_ml.index]

hybrid_nse = calculate_nse(y_val_true, q_hybrid)

print(f"Baseline NSE: 0.44")
print(f"Hybrid NSE: {hybrid_nse}")
# # print(f"Improvement: +{hybrid_nse - 0.44:.3f}")


Baseline NSE: 0.44
Hybrid NSE: 0.9471751433220192


In [21]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    max_depth  = 5,
    learning_rate = 0.05,
    n_estimators = 1500,
)

# ✅ USE THE FIXED DATAFRAME
train_ml = df_adv_fixed[:'2017-12-31']
val_ml = df_adv_fixed['2018-01-01':'2021-12-31']
test_ml = df_adv_fixed['2022-01-01':]

# Train XGBoost on residuals
X_train_ml = train_ml.drop(['residuals'], axis = 1)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(['residuals'], axis = 1)
y_val_ml = val_ml['residuals'] # Use the residuals from the same clean dataframe

X_test_ml = test_ml.drop(['residuals'], axis = 1)
y_test_ml = test_ml['residuals']

xgb.fit(X_train_ml, y_train_ml)
residual_pred = xgb.predict(X_val_ml)
q_hybrid = residual_pred + val_ml['baseline_q']
# ✅ IMPORTANT: Use the correct ground truth for comparison
# Y_val_lagged has the wrong index. We need the 'runoff' that corresponds to val_ml.
# We can get this from the original Y_full dataframe by slicing it with the val_ml index.
y_val_true = x_lagged['runoff'].loc[val_ml.index]

hybrid_nse = calculate_nse(y_val_true, q_hybrid)

print(f"Baseline NSE: 0.44")
print(f"Hybrid NSE: {hybrid_nse}")
# # print(f"Improvement: +{hybrid_nse - 0.44:.3f}")

Baseline NSE: 0.44
Hybrid NSE: 0.9471751433220192


In [ ]:
df_adv.describe()

In [22]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    max_depth  = 5,
    learning_rate = 0.05,
    n_estimators = 1500,
)

# # ✅ USE THE CLEANED DATAFRAME
# train_ml = df_adv[:'2017-12-31']
# val_ml = df_adv['2018-01-01':'2021-12-31']
test_ml = df_adv_fixed['2022-01-01':]

# Train XGBoost on residuals
X_train_ml = train_ml.drop(['residuals'], axis = 1)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(['residuals'], axis = 1)
y_val_ml = val_ml['residuals'] # Use the residuals from the same clean dataframe

X_test_ml = test_ml.drop(['residuals'], axis = 1)
y_test_ml = test_ml['residuals']

xgb.fit(X_train_ml, y_train_ml)
residual_pred = xgb.predict(X_val_ml)
q_hybrid = residual_pred + val_ml['baseline_q']
# ✅ IMPORTANT: Use the correct ground truth for comparison
# Y_val_lagged has the wrong index. We need the 'runoff' that corresponds to val_ml.
# We can get this from the original Y_full dataframe by slicing it with the val_ml index.
y_val_true = x_lagged['runoff'].loc[val_ml.index]

hybrid_nse = calculate_nse(y_val_true, q_hybrid)

print(f"Baseline NSE: 0.44")
print(f"Hybrid NSE: {hybrid_nse}")
# # print(f"Improvement: +{hybrid_nse - 0.44:.3f}")


Baseline NSE: 0.44
Hybrid NSE: 0.9471751433220192


In [ ]:
df_adv.head()

In [23]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    max_depth  = 5,
    learning_rate = 0.05,
    n_estimators = 1500,
)

# ✅ USE THE CLEANED DATAFRAME
train_ml = df_adv[:'2017-12-31']
val_ml = df_adv['2018-01-01':'2021-12-31']
test_ml = df_adv['2022-01-01':]

# Train XGBoost on residuals
X_train_ml = train_ml.drop(['residuals'], axis = 1)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(['residuals'], axis = 1)
y_val_ml = val_ml['residuals'] # Use the residuals from the same clean dataframe

X_test_ml = test_ml.drop(['residuals'], axis = 1)
y_test_ml = test_ml['residuals']

xgb.fit(X_train_ml, y_train_ml)
residual_pred = xgb.predict(X_val_ml)
q_hybrid = residual_pred + val_ml['baseline_q']
# ✅ IMPORTANT: Use the correct ground truth for comparison
# Y_val_lagged has the wrong index. We need the 'runoff' that corresponds to val_ml.
# We can get this from the original Y_full dataframe by slicing it with the val_ml index.
y_val_true = x_lagged['runoff'].loc[val_ml.index]

hybrid_nse = calculate_nse(y_val_true, q_hybrid)

print(f"Baseline NSE: 0.44")
print(f"Hybrid NSE: {hybrid_nse}")
# # print(f"Improvement: +{hybrid_nse - 0.44:.3f}")


Baseline NSE: 0.44
Hybrid NSE: 0.9541236331672822


In [ ]:
evaluate_model(y_val_true, q_hybrid)

In [24]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    max_depth  = 5,
    learning_rate = 0.05,
    n_estimators = 1500,
)

# ✅ USE THE CLEANED DATAFRAME
train_ml = df_adv[:'2017-12-31']
val_ml = df_adv['2018-01-01':'2021-12-31']
test_ml = df_adv['2022-01-01':]

# Train XGBoost on residuals
X_train_ml = train_ml.drop(['residuals'], axis = 1)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(['residuals'], axis = 1)
y_val_ml = val_ml['residuals'] # Use the residuals from the same clean dataframe

X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
y_train_val_ml = pd.concat([y_train_ml, y_val_ml])

X_test_ml = test_ml.drop(['residuals'], axis = 1)
y_test_ml = test_ml['residuals']

xgb.fit(X_train_val_ml, y_train_val_ml)
residual_pred = xgb.predict(X_test_ml)
q_hybrid = residual_pred + test_ml['baseline_q']

# ✅ IMPORTANT: Use the correct ground truth for comparison
# Y_val_lagged has the wrong index. We need the 'runoff' that corresponds to val_ml.
# We can get this from the original Y_full dataframe by slicing it with the val_ml index.
y_test_true = x_lagged['runoff'].loc[test_ml.index]

hybrid_nse = calculate_nse(y_test_true, q_hybrid)

print(f"Baseline NSE: 0.44")
print(f"Hybrid NSE: {hybrid_nse}")
# # print(f"Improvement: +{hybrid_nse - 0.44:.3f}")


Baseline NSE: 0.44
Hybrid NSE: 1.0


In [ ]:
# X_train_ml = train_ml.drop(['residuals'])
train_ml.head()

In [ ]:
print("Original df_full length:", len(df_full))
print("Cleaned df_full length:", len(df_full_clean))


In [ ]:
df_adv = create_lag_adv(df1)
df_adv['baseline_q'] = model.predict(df_full)
df_adv['residuals'] = Y_full['runoff'] - df_adv['baseline_q']

In [ ]:
df_adv.head()

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

y_test_pred = xgb.predict(X_test_lagged)
rmse = np.sqrt(mean_squared_error(Y_test_lagged, y_test_pred))
print(rmse)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(y_true, y_pred):
    """
    Calculate multiple regression metrics
    
    Parameters:
    -----------
    y_true : array-like
        Actual observed values
    y_pred : array-like
        Model predicted values
    
    Returns:
    --------
    dict : Dictionary containing all metrics
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    # RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # MAE
    mae = mean_absolute_error(y_true, y_pred)
    
    # R²
    r2 = r2_score(y_true, y_pred)
    
    # NSE (Nash-Sutcliffe Efficiency)
    mean_obs = np.mean(y_true)
    nse = 1 - (np.sum((y_true - y_pred) ** 2) / 
               np.sum((y_true - mean_obs) ** 2))
    
    # RMSE as % of mean
    rmse_percent = (rmse / np.mean(y_true)) * 100
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'NSE': nse,
        'RMSE (%)': rmse_percent
    }

# Usage:
y_test_pred = model.predict(X_test_lagged)
metrics = evaluate_model(Y_test_lagged, y_test_pred)

print("Model Performance Metrics:")
print("-" * 40)
for metric, value in metrics.items():
    print(f"{metric:12s}: {value:.4f}")

In [ ]:
# # same as above

# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LinearRegression
# lag_scaler = StandardScaler()


# x_lagged = df1.copy()

# x_train = x_lagged[:'2017-12-31']
# x_val = x_lagged['2018-01-01':'2021-12-31']
# x_test = x_lagged['2022-01-01':]

# x_train = create_lag(x_train, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)

# x_val =create_lag(x_val, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)

# x_test = create_lag(x_test, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)


# y_train = x_train[:'2017-12-31']
# y_val = x_val['2018-01-01':'2021-12-31']
# y_test = x_test['2022-01-01':]

# X_train_lagged = x_train[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
# Y_train_lagged = y_train[['runoff']]
# Y_val_lagged = y_val[['runoff']]
# X_val_lagged = x_val[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
# X_test_lagged = x_test[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
# Y_test_lagged = x_test[['runoff']]

# lag_scaler.fit(X_train_lagged)
 
# x_train_scaled = lag_scaler.transform(X_train_lagged)
# x_val_scaled = lag_scaler.transform(X_val_lagged)
# x_test_scaled = lag_scaler.transform(X_test_lagged)

# amodel = LinearRegression()
# amodel.fit(X_train_lagged, Y_train_lagged)
# amodel.fit(X_val_lagged, Y_val_lagged)

# y_val_pred = amodel.predict(X_test_lagged)
# baseline_nse = calculate_nse(Y_test_lagged, y_val_pred)
# print(baseline_nse)
# # alpha, beta, gamma, delta=amodel.coef_
# print(amodel.coef_)

In [ ]:
# #split after, create lags first
# from sklearn.linear_model import LinearRegression
# x_lagged = df1.copy()
# # Your splits (already correct)
# x_train = x_lagged[:'2017-12-31'].copy()
# x_val = x_lagged['2018-01-01':'2021-12-31'].copy()
# x_test = x_lagged['2022-01-01':].copy()

# # Create lags (already correct)
# x_train = create_lag(x_train, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)
# x_val = create_lag(x_val, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)
# x_test = create_lag(x_test, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)

# # Prepare features
# X_train = x_train[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
# Y_train = x_train[['runoff']]

# X_val = x_val[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
# Y_val = x_val[['runoff']]

# X_test = x_test[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
# Y_test = x_test[['runoff']]

# # CORRECTED: Fit on train+val ONCE
# X_train_val = pd.concat([X_train, X_val])
# Y_train_val = pd.concat([Y_train, Y_val])

# amodel = LinearRegression()
# amodel.fit(X_train_val, Y_train_val)  # Single fit on combined data

# # Predict on test
# y_test_pred = amodel.predict(X_test)

# # Evaluate
# baseline_nse = calculate_nse(Y_test, y_test_pred)
# print(f"Test NSE: {baseline_nse}")
# print(f"Coefficients: {amodel.coef_}")

In [ ]:
#accumulated precipitation
#5 month - 11->3, 3 month window - 12-2
# df1['precipitation_acc'] = df1['precipitation']
# for precip in range(2000,2026):
#     df1['precipitation_acc'] = df1['precipitation'] 
df1.index = pd.to_datetime(df1.index)

winter_months = df1[df1.index.month.isin([11, 12, 1, 2, 3])]
print(winter_months['precipitation'].head())
print(f"Total winter records: {len(winter_months)}")

In [ ]:
# #not honest
# from sklearn.linear_model import LinearRegression
# import pandas as pd

# def evaluate_config(lag_precip, lag_sca, lag_dd, lag_et):
#     """Test a lag configuration"""
#     x_lagged= df1.copy()
#     # Split
#     x_train = x_lagged[:'2017-12-31'].copy()
#     x_val = x_lagged['2018-01-01':'2021-12-31'].copy()
#     x_test = x_lagged['2022-01-01':].copy()
    
#     # Create lags
#     x_train = create_lag(x_train, lag_precip, lag_dd, lag_sca, lag_et)
#     x_val = create_lag(x_val, lag_precip, lag_dd, lag_sca, lag_et)
#     x_test = create_lag(x_test, lag_precip, lag_dd, lag_sca, lag_et)
    
#     # Features
#     feature_cols = ['precipitation', 'precipitation_lagged_cum', 
#                     'melt_proxy', 'et_loss_lagged']
    
#     X_train = x_train[feature_cols]
#     Y_train = x_train[['runoff']]
#     X_val = x_val[feature_cols]
#     Y_val = x_val[['runoff']]
#     X_test = x_test[feature_cols]
#     Y_test = x_test[['runoff']]
    
#     # Train on train+val
#     X_train_val = pd.concat([X_train, X_val])
#     Y_train_val = pd.concat([Y_train, Y_val])
    
#     model = LinearRegression()
#     model.fit(X_train_val, Y_train_val)
    
#     # Evaluate
#     Y_test_pred = model.predict(X_test)
#     test_nse = calculate_nse(Y_test, Y_test_pred)
    
#     return test_nse

# # Test these configurations
# configs = [
#     # Your current best
#     {'precip': 1, 'sca': 2, 'dd': 6, 'et': 1, 'name': 'Current'},
    
#     # Physically constrained (same lag for S and DD)
#     {'precip': 1, 'sca': 2, 'dd': 2, 'et': 1, 'name': 'Constrained 2,2'},
#     {'precip': 1, 'sca': 3, 'dd': 3, 'et': 1, 'name': 'Constrained 3,3'},
#     {'precip': 1, 'sca': 4, 'dd': 4, 'et': 1, 'name': 'Constrained 4,4'},

#     # Compromise options
#     {'precip': 1, 'sca': 2, 'dd': 4, 'et': 1, 'name': 'Compromise 2,4'},
#     {'precip': 1, 'sca': 2, 'dd': 4, 'et': 1, 'name': 'With precip 2,4'},
# ]

# print("Configuration Testing:")
# print("-" * 60)
# for cfg in configs:
#     nse = evaluate_config(
#         lag_precip=cfg['precip'],
#         lag_sca=cfg['sca'],
#         lag_dd=cfg['dd'],
#         lag_et=cfg['et']
#     )
#     print(f"{cfg['name']:20} | Test NSE: {nse}")

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
x_lagged= create_lag(df1, lag_precip=1, lag_dd=6, lag_sca=2, lag_et=1)
x_train = x_lagged[:'2017-12-31'].copy()
x_val = x_lagged['2018-01-01':'2021-12-31'].copy()
x_test = x_lagged['2022-01-01':].copy()
def evaluate_config(lag_precip, lag_sca, lag_dd, lag_et):
    """Test a lag configuration"""
    # Split
    # x_train = x_lagged[:'2017-12-31'].copy()
    # x_val = x_lagged['2018-01-01':'2021-12-31'].copy()
    # x_test = x_lagged['2022-01-01':].copy()
    
    # Create lags
    # x_train = create_lag(x_train, lag_precip, lag_dd, lag_sca, lag_et)
    # x_val =/ create_lag(x_val, lag_precip, lag_dd, lag_sca, lag_et)
    # x_test = create_lag(x_test, lag_precip, lag_dd, lag_sca, lag_et)
    
    # Features
    feature_cols = ['precipitation', 'precipitation_lagged_cum', 
                    'melt_proxy', 'et_loss_lagged']
    
    X_train = x_train[feature_cols]
    Y_train = x_train[['runoff']]
    X_val = x_val[feature_cols]
    Y_val = x_val[['runoff']]
    X_test = x_test[feature_cols]
    Y_test = x_test[['runoff']]
    
    # Train on train+val
    X_train_val = pd.concat([X_train, X_val])
    Y_train_val = pd.concat([Y_train, Y_val])
    
    model = LinearRegression()
    model.fit(X_train_val, Y_train_val)
    
    # Evaluate
    Y_test_pred = model.predict(X_test)
    test_nse = calculate_nse(Y_test, Y_test_pred)
    
    return test_nse

# Test these configurations
configs = [
    # Your current best
    {'precip': 1, 'sca': 2, 'dd': 6, 'et': 1, 'name': 'Current'},
    
    # Physically constrained (same lag for S and DD)
    {'precip': 1, 'sca': 2, 'dd': 2, 'et': 1, 'name': 'Constrained 2,2'},
    {'precip': 1, 'sca': 3, 'dd': 3, 'et': 1, 'name': 'Constrained 3,3'},
    {'precip': 1, 'sca': 4, 'dd': 4, 'et': 1, 'name': 'Constrained 4,4'},

    # Compromise options
    {'precip': 1, 'sca': 2, 'dd': 4, 'et': 1, 'name': 'Compromise 2,4'},
    {'precip': 1, 'sca': 2, 'dd': 4, 'et': 1, 'name': 'With precip 2,4'},
]

print("Configuration Testing:")
print("-" * 60)
for cfg in configs:
    nse = evaluate_config(
        lag_precip=cfg['precip'],
        lag_sca=cfg['sca'],
        lag_dd=cfg['dd'],
        lag_et=cfg['et']
    )
    print(f"{cfg['name']:20} | Test NSE: {nse}")

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LinearRegression
# from sklearn.preprocessing import StandardScaler
# from itertools import product

# def create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et):
#     """Create lagged features with different lags for each variable"""
#     df_copy = df.copy()
    
#     df_copy['precipitation_lagged'] = df_copy['precipitation'].shift(lag_precip)
#     df_copy['sca_lagged'] = df_copy['sca'].shift(lag_sca)
#     df_copy['dd_lagged'] = df_copy['dd'].shift(lag_dd)
#     df_copy['et_loss_lagged'] = df_copy['et_loss'].shift(lag_et)
#     df_copy['melt_proxy'] = df_copy['sca_lagged'] * df_copy['dd_lagged']
    
#     df_clean = df_copy.dropna()
#     return df_clean

# def calculate_nse(observed, predicted):
#     mean_obs = np.mean(observed)
#     numerator = np.sum((observed - predicted) ** 2)
#     denominator = np.sum((observed - mean_obs) ** 2)
#     return 1 - (numerator / denominator)

# # Load data
# df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
# df = df.set_index('date')
# df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'], inplace=True, errors='ignore')

# # Test all combinations
# results = []

# # Generate all combinations: 0-7 for each feature = 8^4 = 4096 combinations
# # lag_range = range(1, 8)
# for lag_precip, lag_sca, lag_dd, lag_et in product(range(0,1),
#                                                    range(0,3),
#                                                    range(0,2),
#                                                    range(0,2)

#     ):
#     try:
#         # Create lagged features
#         x = create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et)
        
#         # Select features and target
#         X = x[['precipitation','precipitation_lagged', 'melt_proxy', 'et_loss_lagged']]
#         y = x['runoff']
        
#         # Split data
#         X_train = X[:'2017-12-31']
#         X_val = X['2018-01-01':'2021-12-31']
        
#         y_train = y[:'2017-12-31']
#         y_val = y['2018-01-01':'2021-12-31']
        
#         # Skip if not enough data
#         if len(X_train) < 10 or len(X_val) < 10:
#             continue
        
#         # Scale features
#         scaler = StandardScaler()
#         X_train_scaled = scaler.fit_transform(X_train)
#         X_val_scaled = scaler.transform(X_val)
        
#         # Train model
#         model = LinearRegression()
#         model.fit(X_train_scaled, y_train)
        
#         # Predict and evaluate
#         y_val_pred = model.predict(X_val_scaled)
#         nse = calculate_nse(y_val, y_val_pred)
        
#         # Store results
#         results.append({
#             'lag_precip': lag_precip,
#             'lag_sca': lag_sca,
#             'lag_dd': lag_dd,
#             'lag_et': lag_et,
#             'nse': nse
#         })
#     except Exception as e:
#         continue

# # Convert to DataFrame
# results_df = pd.DataFrame(results)
# results_df.head()
# # Find best combination
# best_result = results_df.loc[results_df['nse'].idxmax()]

# print(f"\nBest NSE: {best_result['nse']:.4f}")
# print(f"Best lag combination:")
# print(f"  Precipitation lag: {int(best_result['lag_precip'])}")
# print(f"  SCA lag: {int(best_result['lag_sca'])}")
# print(f"  DD lag: {int(best_result['lag_dd'])}")
# print(f"  ET loss lag: {int(best_result['lag_et'])}")

# # Save all results
# results_df.to_csv('lag_optimization_results.csv', index=False)

# # Show top 10 combinations
# print("\nTop 10 lag combinations:")
# print(results_df.nlargest(50, 'nse'))

In [ ]:
x_lagged_x = create_lag(df1, lag_precip=1, lag_dd=4, lag_sca=3, lag_et=1)
print(x_lagged_x[['precipitation','precipitation_lagged_cum','precipitation_lagged','sca_lagged','dd_lagged', 'runoff']].corr()['runoff'])


In [ ]:
# Run this and tell me the output
print("Data summary:")
print(x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].describe())

print("\nCorrelations with discharge:")
print(x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff'])

print("\nFirst 5 rows:")
print(x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].head())

In [ ]:
corr_matrix = x.corr()
corr_matrix

In [ ]:
x_lagged

In [ ]:
corr = x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff']
print(corr)

In [ ]:
print("SCA seasonal pattern:")
x_lagged.set_index('date')
print(x_lagged.groupby(x_lagged.index.month)['SCA'].mean())
   # If SCA is HIGH in winter, LOW in summer → it's snow cover %
   # This is your problem!

In [ ]:
# Plot heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LinearRegression
# from sklearn.preprocessing import StandardScaler
# from itertools import product
# # MAX_LAG = 7
# # df = df.iloc[MAX_LAG:]


# def create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et):
#     """Create lagged features with different lags for each variable"""
#     df_copy = df.copy()
    
#     df_copy['precipitation_lagged'] = df_copy['precipitation'].shift(lag_precip)
#     df_copy['sca_lagged'] = df_copy['sca'].shift(lag_sca)
#     df_copy['dd_lagged'] = df_copy['dd'].shift(lag_dd)
#     df_copy['et_loss_lagged'] = df_copy['et_loss'].shift(lag_et)
#     df_copy['melt_proxy'] = df_copy['sca_lagged'] * df_copy['dd_lagged']
    
#     df_clean = df_copy.dropna()
#     return df_clean

# def calculate_nse(observed, predicted):
#     mean_obs = np.mean(observed)
#     numerator = np.sum((observed - predicted) ** 2)
#     denominator = np.sum((observed - mean_obs) ** 2)
#     return 1 - (numerator / denominator)

# # Load data
# df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
# df = df.set_index('date')
# df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'], inplace=True, errors='ignore')

# # Test all combinations
# results = []

# # Generate all combinations: 0-7 for each feature = 8^4 = 4096 combinations
# lag_range = range(1, 8)

# for lag_precip, lag_sca, lag_dd, lag_et in product(lag_range, repeat=4):
#     try:
#         # Create lagged features
#         x = create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et)
        
#         # Select features and target
#         X = x[['precipitation_lagged', 'melt_proxy', 'et_loss_lagged']]
#         y = x['runoff']
        
#         # Split data
#         X_train = X[:'2017-12-31']
#         X_val = X['2018-01-01':'2021-12-31']
        
#         y_train = y[:'2017-12-31']
#         y_val = y['2018-01-01':'2021-12-31']
        
#         # Skip if not enough data
#         if len(X_train) < 10 or len(X_val) < 10:
#             continue
        
#         # Scale features
#         scaler = StandardScaler()
#         X_train_scaled = scaler.fit_transform(X_train)
#         X_val_scaled = scaler.transform(X_val)
        
#         # Train model
#         model = LinearRegression()
#         model.fit(X_train_scaled, y_train)
        
#         # Predict and evaluate
#         y_val_pred = model.predict(X_val_scaled)
#         nse = calculate_nse(y_val, y_val_pred)
        
#         # Store results
#         results.append({
#             'lag_precip': lag_precip,
#             'lag_sca': lag_sca,
#             'lag_dd': lag_dd,
#             'lag_et': lag_et,
#             'nse': nse
#         })
        
#     except Exception as e:
#         continue

# # Convert to DataFrame
# results_df = pd.DataFrame(results)
# results_df.head()
# # Find best combination
# best_result = results_df.loc[results_df['nse'].idxmax()]

# print(f"\nBest NSE: {best_result['nse']:.4f}")
# print(f"Best lag combination:")
# print(f"  Precipitation lag: {int(best_result['lag_precip'])}")
# print(f"  SCA lag: {int(best_result['lag_sca'])}")
# print(f"  DD lag: {int(best_result['lag_dd'])}")
# print(f"  ET loss lag: {int(best_result['lag_et'])}")

# # Save all results
# results_df.to_csv('lag_optimization_results.csv', index=False)

# # Show top 10 combinations
# print("\nTop 10 lag combinations:")
# print(results_df.nlargest(50, 'nse'))

In [ ]:
x_lagged_x = create_lag(df1, lag_precip=1, lag_dd=4, lag_sca=2, lag_et=1)
print(x_lagged_x[['melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff'])
